In [1]:
import gc
# 强制进行垃圾回收
gc.collect()

16

In [3]:
import pandas as pd
import numpy as np
import scipy.sparse as sp
from sklearn.model_selection import train_test_split

# 加载原始数据
df = pd.read_csv("dataset/data_train.csv")

# 1234 随机数分割，确保和训练模型时一致
train_df, val_df_total = train_test_split(df, test_size=0.2, random_state=1234)

# 构建 URM_train (80%)，模型是基于这个训练的
URM_train = sp.csr_matrix((np.ones(len(train_df)), (train_df['row'], train_df['col'])))
n_users, n_items = URM_train.shape

In [97]:
from Recommenders.SLIM.SLIMElasticNetRecommender import SLIMElasticNetRecommender
from Recommenders.Neural.MultVAE_PyTorch_Recommender import MultVAERecommender_PyTorch_OptimizerMask
from Recommenders.MatrixFactorization.IALSRecommender import IALSRecommender

temp_folder = "temp_output"

# 加载 SLIM
slim = SLIMElasticNetRecommender(URM_train)
slim.load_model(temp_folder, file_name="SLIMElasticNetRecommender_best_model_best")

# 加载 MultVAE
vae = MultVAERecommender_PyTorch_OptimizerMask(URM_train)
vae.load_model(temp_folder, file_name="multvae_best_trial_14_recall_0.253894-145")

# 加载 80% 训练好的 IALS 模型 (用于本地验证)
ials = IALSRecommender(URM_train)
ials.load_model(temp_folder, file_name="IALSRecommender_best_model_best")

SLIMElasticNetRecommender: Loading model from file 'temp_outputSLIMElasticNetRecommender_best_model_best'
SLIMElasticNetRecommender: Loading complete
IALSRecommender: Loading model from file 'temp_outputIALSRecommender_best_model_best'
IALSRecommender: Loading complete


In [98]:
def build_three_model_rank_dataset(recommender_list, interaction_df, n_negatives=5):
    slim_rec, vae_rec, ials_rec = recommender_list
    unique_users = interaction_df['row'].unique()
    n_items = slim_rec.URM_train.shape[1]

    all_features = []
    all_labels = []

    print(f"正在构建三模型特征... 处理用户数: {len(unique_users)}")

    for u in unique_users:
        # 批量获取三个模型的分数
        s_s = slim_rec._compute_item_score([u])[0]
        s_v = vae_rec._compute_item_score([u])[0]
        s_i = ials_rec._compute_item_score([u])[0]

        # 计算全量排名 (1, 2, 3...)
        r_s = np.empty(n_items, dtype=np.int32)
        r_s[np.argsort(-s_s)] = np.arange(n_items) + 1

        r_v = np.empty(n_items, dtype=np.int32)
        r_v[np.argsort(-s_v)] = np.arange(n_items) + 1

        r_i = np.empty(n_items, dtype=np.int32)
        r_i[np.argsort(-s_i)] = np.arange(n_items) + 1

        # 提取验证集正样本
        pos_items = interaction_df[interaction_df['row'] == u]['col'].values

        for item_idx in pos_items:
            # 特征：[SLIM_Rank, VAE_Rank, IALS_Rank]
            feat = [r_s[item_idx], r_v[item_idx], r_i[item_idx]]
            all_features.append(feat); all_labels.append(1)

            # 负采样
            for _ in range(n_negatives):
                neg_i = np.random.randint(0, n_items)
                while slim_rec.URM_train[u, neg_i] > 0:
                    neg_i = np.random.randint(0, n_items)
                feat_neg = [r_s[neg_i], r_v[neg_i], r_i[neg_i]]
                all_features.append(feat_neg); all_labels.append(0)

    return np.array(all_features), np.array(all_labels)

# 执行构建
X_3_raw, y_3 = build_three_model_rank_dataset([slim, vae, ials], val_df_total)

正在构建三模型特征... 处理用户数: 27059


In [102]:
np.save("X_meta_train.npy", X_3_raw)
np.save("y_meta_train.npy", y_3)
# 下次直接 load，不需要再跑 build 逻辑

In [103]:
X_train = np.load("X_meta_train.npy")
y_train = np.load("y_meta_train.npy")

In [99]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# 1. 转换特征为倒数排名 1/(Rank+1)
X_3_inv = 1.0 / (X_3_raw + 1.0)

# 2. 划分训练/测试 (Meta-level)
X_m_train, X_m_test, y_m_train, y_m_test = train_test_split(
    X_3_inv, y_3, test_size=0.20, random_state=42
)

# 3. 训练 LR (不归一化版本，保持之前的稳健逻辑)
lr_3_meta = LogisticRegression(C=1.0)
lr_3_meta.fit(X_m_train, y_m_train)

# 4. 查看权重分配
print("[三模型融合 - 权重分析]")
model_names = ["SLIM_InvRank", "VAE_InvRank", "IALS_InvRank"]
for name, coef in zip(model_names, lr_3_meta.coef_[0]):
    print(f"{name:15}: {coef:.4f}")

# 5. 计算 Recall@20
# (此处调用之前的 local_validation 逻辑，只需在特征构造处增加 IALS 列即可)

[三模型融合 - 权重分析]
SLIM_InvRank   : 82.0784
VAE_InvRank    : 129.4904
IALS_InvRank   : 115.6085


In [101]:
def local_validation_recall_3_models(recommender_list, val_df, meta_model, n_users=1000):
    """
    完成第 5 步：计算三模型融合后的本地 Recall@20
    """
    slim_rec, vae_rec, ials_rec = recommender_list
    n_items = slim_rec.URM_train.shape[1]

    # 随机抽取部分验证集用户进行评估，以节省计算时间
    test_users = val_df['row'].unique()
    np.random.seed(42)
    test_users = np.random.choice(test_users, min(len(test_users), n_users), replace=False)

    recalls = []

    print(f"正在验证 {len(test_users)} 个用户的 Recall@20...")

    for u in test_users:
        # 1. 获取该用户在验证集中的真实交互物品 (Ground Truth)
        gt = val_df[val_df['row'] == u]['col'].values

        # 2. 获取三个模型的基础分数
        s_s = slim_rec._compute_item_score([u])[0]
        s_v = vae_rec._compute_item_score([u])[0]
        s_i = ials_rec._compute_item_score([u])[0]

        # 3. 粗排候选集：取三个模型各自 Top 500 的并集（决赛圈）
        c_s = np.argpartition(-s_s, 500)[:500]
        c_v = np.argpartition(-s_v, 500)[:500]
        c_i = np.argpartition(-s_i, 500)[:500]
        candidates = np.unique(np.concatenate([c_s, c_v, c_i]))

        # 4. 计算全量物品的整数排名 (1, 2, 3...)
        r_s_all = np.empty(n_items, dtype=np.int32)
        r_s_all[np.argsort(-s_s)] = np.arange(n_items) + 1

        r_v_all = np.empty(n_items, dtype=np.int32)
        r_v_all[np.argsort(-s_v)] = np.arange(n_items) + 1

        r_i_all = np.empty(n_items, dtype=np.int32)
        r_i_all[np.argsort(-s_i)] = np.arange(n_items) + 1

        # 5. 构造候选集的倒数排名特征矩阵 [1/Rank_S, 1/Rank_V, 1/Rank_I]
        # 注意：这里的列顺序必须与你训练 lr_3_meta 时完全一致
        X_cand = np.column_stack([
            1.0 / (r_s_all[candidates] + 1.0),
            1.0 / (r_v_all[candidates] + 1.0),
            1.0 / (r_i_all[candidates] + 1.0)
        ])

        # 6. 使用逻辑回归预测最终得分
        final_scores = meta_model.predict_proba(X_cand)[:, 1]

        # 7. 过滤已购物品 (屏蔽掉训练集 URM_train 中的交互)
        seen_items = slim_rec.URM_train[u].indices
        final_scores[np.isin(candidates, seen_items)] = -np.inf

        # 8. 取 Top 20 并计算 Recall
        top_20_idx = np.argsort(-final_scores)[:20]
        top_20 = candidates[top_20_idx]

        hits = np.isin(gt, top_20).sum()
        recalls.append(hits / len(gt))

    return np.mean(recalls)

# --- 调用执行 ---
mean_recall_3 = local_validation_recall_3_models([slim, vae, ials], val_df_total, lr_3_meta)
print(f"\n[本地验证] 三模型融合最终 Recall@20: {mean_recall_3:.5f}")

正在验证 1000 个用户的 Recall@20...

[本地验证] 三模型融合最终 Recall@20: 0.36187


# 线上提交

In [4]:
# 1. 加载你之前保存好的特征和标签
# X_rank_scaled 应该是 (N_samples, 6) 的矩阵
# y_all 应该是 (N_samples,) 的标签
print("正在加载预存的特征数据...")
X_train = np.load("X_meta_train.npy")
y_train = np.load("y_meta_train.npy")
print(f"特征形状: {X_train.shape}")
print(f"正样本比例: {np.mean(y_train):.2%}")

正在加载预存的特征数据...
特征形状: (3651672, 3)
正样本比例: 16.67%


In [5]:
# 1. 加载 100% 全量数据 (用于模型初始化和已购物品过滤)
print("加载全量训练数据...")
df_all = pd.read_csv("dataset/data_train.csv")
URM_all = sp.csr_matrix((np.ones(len(df_all)), (df_all['row'], df_all['col'])))
n_items = URM_all.shape[1]

加载全量训练数据...


In [7]:
from Recommenders.SLIM.SLIMElasticNetRecommender import SLIMElasticNetRecommender
from Recommenders.Neural.MultVAE_PyTorch_Recommender import MultVAERecommender_PyTorch_OptimizerMask
from Recommenders.MatrixFactorization.IALSRecommender import IALSRecommender

model_folder = "model_output"

# 加载 SLIM
slim_100 = SLIMElasticNetRecommender(URM_all)
slim_100.load_model(model_folder, file_name="5-1final_model_SLIMElasticNetRecommender-better")

# 加载 MultVAE
vae_100  = MultVAERecommender_PyTorch_OptimizerMask(URM_all)
vae_100.load_model(model_folder, file_name="29-MultVAE_Best_Model_Sub")

# 加载 IALS
ials_100 = IALSRecommender(URM_all)
ials_100.load_model(model_folder, file_name="23-final_model_IALSRecommender-best")

SLIMElasticNetRecommender: Loading model from file 'model_output5-1final_model_SLIMElasticNetRecommender-better'
SLIMElasticNetRecommender: Loading complete
IALSRecommender: Loading model from file 'model_output23-final_model_IALSRecommender-best'
IALSRecommender: Loading complete


In [8]:
# 4. 读取目标用户
target_users_df = pd.read_csv("dataset/data_target_users_test.csv")
target_users = target_users_df['user_id'].values

In [9]:
X_3_raw = np.load("X_meta_train.npy")
y_3 = np.load("y_meta_train.npy")

In [13]:
import numpy as np
import joblib
from sklearn.linear_model import LogisticRegression

# 1. 准备全量特征 (基于 80% 模型对 20% 验证集生成的原始数据)
# 假设 X_3_raw 是你之前 build_3_model_rank_dataset 生成的 (N, 3) 矩阵
# y_3 是对应的标签

# 核心变换：转换为 1/(Rank+1)
# 这一步必须在整数排名上执行，以确保非线性权重的敏锐度
X_final_meta = 1.0 / (X_3_raw + 1.0)
y_final_meta = y_3

# 2. 训练最终逻辑回归模型
# 使用 C=1.0 保持标准的正则化强度。如果样本量极大，lbfgs 是最稳的求解器。
final_lr_3_meta = LogisticRegression(C=1.0, solver='lbfgs', max_iter=1000)
final_lr_3_meta.fit(X_final_meta, y_final_meta)

# 3. 打印最终权重（最后的逻辑检查）
print("[最终法官模型 - 权重分配]")
model_names = ["SLIM_InvRank", "VAE_InvRank", "IALS_InvRank"]
weights = final_lr_3_meta.coef_[0]
intercept = final_lr_3_meta.intercept_[0]

for name, w in zip(model_names, weights):
    print(f"{name:15}: {w:.4f}")
print(f"{'Intercept':15}: {intercept:.4f}")

# 4. 持久化保存
# 保存模型后，你可以在任何地方加载它，而不需要重新跑特征构建逻辑
# joblib.dump(final_lr_3_meta, "final_lr_3_model_rank.pkl")

print("\n[成功] 最终 LR 模型已训练并保存为 'final_lr_3_model_rank.pkl'")

[最终法官模型 - 权重分配]
SLIM_InvRank   : 81.4725
VAE_InvRank    : 138.1725
IALS_InvRank   : 118.5370
Intercept      : -2.5758

[成功] 最终 LR 模型已训练并保存为 'final_lr_3_model_rank.pkl'


In [12]:
import pandas as pd
import numpy as np
import joblib

# 1. 确保加载了之前训练好的 LR 法官模型 (lr_3_meta)
lr_3_meta = joblib.load("model_output/32-final_lr_3_model_rank.pkl")

# 2. 设置推理参数
batch_size = 64
n_items = URM_all.shape[1]
submission_results = []

print(f"开始为 {len(target_users)} 个目标用户生成最终提交结果...")

# 3. 开启循环进行批量推理
for i in range(0, len(target_users), batch_size):
    u_batch = target_users[i : i + batch_size]

    # --- A. 批量获取 100% 模型的基础分数 ---
    s_s_batch = slim_100._compute_item_score(u_batch)
    s_v_batch = vae_100. _compute_item_score(u_batch)
    s_i_batch = ials_100._compute_item_score(u_batch)

    for idx, u in enumerate(u_batch):
        s_s = s_s_batch[idx]
        s_v = s_v_batch[idx]
        s_i = s_i_batch[idx]

        # --- B. 候选集提取 (三个模型 Top 500 的并集) ---
        c_s = np.argpartition(-s_s, 500)[:500]
        c_v = np.argpartition(-s_v, 500)[:500]
        c_i = np.argpartition(-s_i, 500)[:500]
        candidates = np.unique(np.concatenate([c_s, c_v, c_i]))

        # --- C. 计算全量排名 (用于特征构造) ---
        # 注意：这里必须要针对 100% 模型算出的全量分进行 argsort
        r_s_all = np.empty(n_items, dtype=np.int32)
        r_s_all[np.argsort(-s_s)] = np.arange(n_items) + 1

        r_v_all = np.empty(n_items, dtype=np.int32)
        r_v_all[np.argsort(-s_v)] = np.arange(n_items) + 1

        r_i_all = np.empty(n_items, dtype=np.int32)
        r_i_all[np.argsort(-s_i)] = np.arange(n_items) + 1

        # --- D. 构造精排特征 [1/Rank_S, 1/Rank_V, 1/Rank_I] ---
        # 严格遵守 80/20 验证时的特征顺序
        X_cand = np.column_stack([
            1.0 / (r_s_all[candidates] + 1.0),
            1.0 / (r_v_all[candidates] + 1.0),
            1.0 / (r_i_all[candidates] + 1.0)
        ])

        # --- E. 逻辑回归打分 ---
        # 获取候选物品为正样本的概率
        final_scores = lr_3_meta.predict_proba(X_cand)[:, 1]

        # --- F. 过滤已购物品 (必须使用 100% 的 URM_all) ---
        seen_items = URM_all[u].indices
        final_scores[np.isin(candidates, seen_items)] = -np.inf

        # --- G. 截断 Top 20 ---
        top_20_idx = np.argsort(-final_scores)[:20]
        final_recs = candidates[top_20_idx]

        # 保存格式：user_id, item_list(空格分隔)
        submission_results.append([u, " ".join(map(str, final_recs))])

    if i % (batch_size * 10) == 0:
        print(f"已处理 {min(i + batch_size, len(target_users))}/{len(target_users)} 用户...")


开始为 27095 个目标用户生成最终提交结果...
已处理 64/27095 用户...
已处理 704/27095 用户...
已处理 1344/27095 用户...
已处理 1984/27095 用户...
已处理 2624/27095 用户...
已处理 3264/27095 用户...
已处理 3904/27095 用户...
已处理 4544/27095 用户...
已处理 5184/27095 用户...
已处理 5824/27095 用户...
已处理 6464/27095 用户...
已处理 7104/27095 用户...
已处理 7744/27095 用户...
已处理 8384/27095 用户...
已处理 9024/27095 用户...
已处理 9664/27095 用户...
已处理 10304/27095 用户...
已处理 10944/27095 用户...
已处理 11584/27095 用户...
已处理 12224/27095 用户...
已处理 12864/27095 用户...
已处理 13504/27095 用户...
已处理 14144/27095 用户...
已处理 14784/27095 用户...
已处理 15424/27095 用户...
已处理 16064/27095 用户...
已处理 16704/27095 用户...
已处理 17344/27095 用户...
已处理 17984/27095 用户...
已处理 18624/27095 用户...
已处理 19264/27095 用户...
已处理 19904/27095 用户...
已处理 20544/27095 用户...
已处理 21184/27095 用户...
已处理 21824/27095 用户...
已处理 22464/27095 用户...
已处理 23104/27095 用户...
已处理 23744/27095 用户...
已处理 24384/27095 用户...
已处理 25024/27095 用户...
已处理 25664/27095 用户...
已处理 26304/27095 用户...
已处理 26944/27095 用户...


In [14]:
# 4. 导出 CSV 文件
print("正在生成提交文件...")
submission_df = pd.DataFrame(submission_results, columns=['user_id', 'item_list'])
submission_df.to_csv("submission_final_3_models_rank_fusion.csv", index=False)

print("生成成功！文件名: submission_final_3_models_rank_fusion.csv")

正在生成提交文件...
生成成功！文件名: submission_final_3_models_rank_fusion.csv
